# Notebook 02: Cleaning, missing data and univariate EDA

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 2, blocks 2 to 4

NexusHealth · Adamu Onu

You will turn the raw export into an analysis-ready dataset, document every decision in a cleaning log, decide how to handle missing data, and describe each variable. By the end you should be able to defend every row and every variable in your analysis set.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the course folder, wherever this notebook lives inside it.
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Course folder not found. Open the whole course folder in VS Code."

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_REFERENCE = ROOT / "data" / "reference"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"
for folder in (DATA_PROCESSED, OUT_TABLES, OUT_FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
print("Course folder found:", ROOT.name)

# Part A: Data cleaning (block 2)

Cleaning follows five verbs: **audit, standardise, validate, deduplicate, derive**. Each rule is written in code (so it can be re-run) and recorded in a cleaning log (so a reader can judge it).

We read every column as **text** first. That way nothing is silently converted or hidden.

In [ ]:
raw = pd.read_csv(DATA_RAW / "primary_care_raw.csv", dtype=str, keep_default_na=False)
dictionary = pd.read_csv(DATA_RAW / "data_dictionary.csv")
print(raw.shape)
raw.head()

## A1. Audit: look before you change anything

**Categorical variables.** How many spellings of each category are there?

In [ ]:
for col in ["clinic", "sex", "smoking_status"]:
    print(f"\n{col}")
    print(raw[col].map(repr).value_counts().to_string())

`repr` shows quotation marks so you can see stray spaces such as `' North '`.

**Numeric variables.** Which entries cannot be read as numbers?

In [ ]:
NUMERIC = ["age", "bmi", "systolic_bp", "diastolic_bp", "hba1c", "physical_activity_min",
           "medication_adherence", "diabetes", "followup_visits_6m"]
for col in NUMERIC:
    not_numbers = raw.loc[pd.to_numeric(raw[col], errors="coerce").isna(), col]
    if len(not_numbers):
        print(f"{col:<22}", not_numbers.map(repr).value_counts().to_dict())

**Duplicates.** Are any patients recorded more than once?

In [ ]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Repeated patient IDs:", raw["patient_id"].duplicated().sum())
raw[raw["patient_id"].duplicated(keep=False)].sort_values("patient_id").head(8)

**Audit summary.** List the problems you found. Which ones could change a result if ignored?

**Your answer:** _Double-click here and write your answer._

## A2. A cleaning log

Every rule gets one line in the log: what we did and how many values or rows it touched.

In [ ]:
cleaning_log = []

def log_step(rule, n_affected, note=""):
    cleaning_log.append({"step": len(cleaning_log) + 1, "rule": rule,
                         "n_affected": int(n_affected), "note": note})

def n_changed(before, after):
    """Count values that differ, treating two missing values as equal."""
    both_missing = before.isna() & after.isna()
    return int(((before != after) & ~both_missing).sum())

clean = raw.copy()

## A3. Standardise missing-value codes

In [ ]:
MISSING_TOKENS = ["", "NA", "N/A", "Unknown"]
n_tokens = clean.isin(MISSING_TOKENS).sum().sum()
clean = clean.replace(MISSING_TOKENS, np.nan)
log_step("Recode blank, NA, N/A and Unknown to missing", n_tokens, "values")
print("Values recoded to missing:", n_tokens)

## A4. Standardise categories

Trim spaces, use one capitalisation style, and expand abbreviations.

In [ ]:
before = clean["clinic"].copy()
clean["clinic"] = clean["clinic"].str.strip().str.title()
log_step("clinic: trim spaces and use title case", n_changed(before, clean["clinic"]))

before = clean["sex"].copy()
clean["sex"] = clean["sex"].str.strip().str.title().replace({"F": "Female", "M": "Male"})
log_step("sex: trim, title case, F/M to Female/Male", n_changed(before, clean["sex"]))

print(clean["clinic"].value_counts(dropna=False), "\n")
print(clean["sex"].value_counts(dropna=False))

## Exercise 2A (part 1): Standardise smoking status

Apply the same idea to `smoking_status`. Then **check** that exactly three categories remain (`Never`, `Former`, `Current`) and add a line to the cleaning log.

**AI assist.** If you get stuck, ask Copilot: *"In pandas, trim spaces and convert the text column smoking_status in data frame clean to title case. Then show the counts of each value."* Check that the counts add up to 653.

In [ ]:
before = clean["smoking_status"].copy()
clean["smoking_status"] = ...          # your rule here
log_step(...)                          # describe the rule and count the changes

# Check: should print ['Current', 'Former', 'Never']
print(sorted(clean["smoking_status"].dropna().unique()))

## A5. Parse dates

Dates arrived in three formats: `2026-04-17`, `30/04/2026` and `Apr 06, 2026`. Is `02/04/2026` the 2nd of April or the 4th of February? Before assuming, check the evidence: if the first number is ever above 12, the format must be day first.

In [ ]:
slash = clean["visit_date"][clean["visit_date"].str.contains("/", na=False)]
first_part = slash.str.split("/").str[0].astype(int)
second_part = slash.str.split("/").str[1].astype(int)
print("Slash dates:", len(slash), "| first part > 12:", (first_part > 12).sum(),
      "| second part > 12:", (second_part > 12).sum())

The first part is often above 12 and the second never is, so these are day/month/year dates. We parse each format explicitly and stop if anything is left unparsed.

In [ ]:
original = clean["visit_date"].copy()
iso = pd.to_datetime(original, format="%Y-%m-%d", errors="coerce")
day_first = pd.to_datetime(original, format="%d/%m/%Y", errors="coerce")
month_name = pd.to_datetime(original, format="%b %d, %Y", errors="coerce")
clean["visit_date"] = iso.fillna(day_first).fillna(month_name)

assert clean["visit_date"].notna().all(), "Some dates could not be parsed"
log_step("visit_date: parse three formats to ISO dates", (iso.isna()).sum(),
         "non-ISO dates converted; slash dates are day/month/year")
print(clean["visit_date"].min().date(), "to", clean["visit_date"].max().date())

## A6. Convert numbers and validate ranges

Now the missing codes are gone, every numeric column should convert cleanly. We deliberately do **not** use a setting that hides failures: if a strange value remains, we want an error.

Then we apply the valid ranges from the data dictionary. Values outside the range become missing, and every change is logged.

In [ ]:
for col in NUMERIC:
    clean[col] = pd.to_numeric(clean[col])   # fails loudly if anything odd remains

VALID_RANGES = {   # copied from the data dictionary
    "age": (18, 85), "bmi": (10, 70), "systolic_bp": (70, 260), "diastolic_bp": (40, 160),
    "hba1c": (3, 20), "physical_activity_min": (0, 1500), "medication_adherence": (0, 100),
}
for col, (low, high) in VALID_RANGES.items():
    outside = clean[col].notna() & ~clean[col].between(low, high)
    if outside.any():
        bad_values = sorted(clean.loc[outside, col].unique().tolist())
        log_step(f"{col}: values outside {low} to {high} set to missing", outside.sum(),
                 f"values seen: {bad_values}")
        clean.loc[outside, col] = np.nan

pd.DataFrame(cleaning_log)

## Exercise 2A (part 2): Defend a cleaning rule

Pick **one** rule from the log (for example the age rule). Write a four-sentence log entry: (1) what you saw, (2) the rule you applied, (3) how many values it affected, (4) what the rule could get wrong.

**Your answer:** _Double-click here and write your answer._

## A7. Remove duplicates

First exact duplicates (identical in every column). Then, for patients with more than one record, keep the **earliest** visit. Write the rule down *before* you look at which records it removes.

In [ ]:
n_before = len(clean)
clean = clean.drop_duplicates()
log_step("Remove exact duplicate rows", n_before - len(clean), "rows")

repeats = clean[clean["patient_id"].duplicated(keep=False)].sort_values(["patient_id", "visit_date"])
display(repeats)

n_before = len(clean)
clean = clean.sort_values(["patient_id", "visit_date"]).drop_duplicates("patient_id", keep="first")
log_step("Keep the earliest record for repeated patient IDs", n_before - len(clean), "rows")
print("Rows now:", len(clean), "| unique patients:", clean["patient_id"].nunique())

## A8. Derive new variables

* `uncontrolled_bp` (the course-defined outcome): 1 if SBP is 140 or more **or** DBP is 90 or more; 0 if both are known and below those values; missing otherwise. Notice that a patient with SBP 150 and a missing DBP is still classified as uncontrolled.
* `adherence_10`: adherence divided by 10, so a one-unit change means 10 percentage points. This makes regression results easier to read.

In [ ]:
sbp_high = clean["systolic_bp"] >= 140
dbp_high = clean["diastolic_bp"] >= 90
both_known = clean["systolic_bp"].notna() & clean["diastolic_bp"].notna()
clean["uncontrolled_bp"] = np.select([sbp_high | dbp_high, both_known], [1, 0], default=np.nan)
clean["adherence_10"] = clean["medication_adherence"] / 10
log_step("Derive uncontrolled_bp (SBP >= 140 or DBP >= 90)", clean["uncontrolled_bp"].notna().sum(),
         "teaching definition only")
log_step("Derive adherence_10 = medication_adherence / 10", clean["adherence_10"].notna().sum())

# A quick check of the derived outcome on a few rows
clean[["systolic_bp", "diastolic_bp", "uncontrolled_bp"]].sample(6, random_state=1)

## A9. Validate, then save

Assertions are tests that stop the notebook if a rule has been broken. They protect you when you re-run everything later, or when someone sends you an updated export.

In [ ]:
assert clean["patient_id"].is_unique
assert set(clean["clinic"].dropna()) <= {"North", "East", "South", "West"}
assert set(clean["sex"].dropna()) <= {"Female", "Male"}
assert set(clean["smoking_status"].dropna()) <= {"Never", "Former", "Current"}
for col, (low, high) in VALID_RANGES.items():
    assert clean[col].dropna().between(low, high).all(), col
assert set(clean["uncontrolled_bp"].dropna()) <= {0, 1}

# Whole-number variables: store them as integers that allow missing values
for col in ["age", "diabetes", "followup_visits_6m", "uncontrolled_bp"]:
    clean[col] = clean[col].astype("Int64")

clean.to_csv(DATA_PROCESSED / "primary_care_clean.csv", index=False, date_format="%Y-%m-%d")
pd.DataFrame(cleaning_log).to_csv(OUT_TABLES / "cleaning_log.csv", index=False)
print("All checks passed. Saved", len(clean), "patients.")

# Part B: Missing data (block 3)

| Mechanism | Plain-language meaning | Example here |
|---|---|---|
| **MCAR** (missing completely at random) | Missingness has nothing to do with the patient | A form page lost at random |
| **MAR** (missing at random) | Missingness depends on things we *did* record | HbA1c less often measured in people without diabetes |
| **MNAR** (missing not at random) | Missingness depends on the missing value itself | People with poor adherence skip the adherence questions |

We can never prove which mechanism applies from the data alone. We can look for clues, choose a reasonable approach, and be honest about what it assumes.

## B1. How much is missing?

In [ ]:
missing_table = pd.DataFrame({
    "n_missing": clean.isna().sum(),
    "pct_missing": (clean.isna().mean() * 100).round(1),
}).sort_values("n_missing", ascending=False)
missing_table[missing_table["n_missing"] > 0]

## B2. Looking for clues

Is HbA1c missing more often in some patients than others?

In [ ]:
clean["hba1c_missing"] = clean["hba1c"].isna()
print((clean.groupby("diabetes")["hba1c_missing"].mean() * 100).round(1).rename("% HbA1c missing by diabetes"))
print()
print((clean.groupby("clinic")["hba1c_missing"].mean() * 100).round(1).rename("% HbA1c missing by clinic"))

Now the main exposure. Do people with missing adherence look different on what we *did* record?

In [ ]:
clean["adherence_missing"] = clean["medication_adherence"].isna()
clean.groupby("adherence_missing").agg(
    n=("patient_id", "size"),
    mean_age=("age", "mean"),
    pct_diabetes=("diabetes", lambda s: 100 * s.mean()),
    mean_sbp=("systolic_bp", "mean"),
).round(1)

**What do these clues suggest?** For HbA1c and for adherence, say which mechanism looks plausible and why you cannot be certain.

**Your answer:** _Double-click here and write your answer._

## B3. Build the analysis set (complete-case analysis)

Our models need the outcome (SBP and DBP), the exposure (adherence) and the covariates (age, sex, BMI, diabetes, smoking). A **complete-case** analysis keeps only patients with all of these. It is simple and transparent, and it is a reasonable *starting point*. It can be biased if missingness depends on the outcome, and it always costs precision. Multiple imputation is the usual next step and needs further training.

We record how many patients are lost at each step, in the order the variables matter to the question.

In [ ]:
flow_steps = [
    ("Unique patients after cleaning", []),
    ("Outcome available (SBP and DBP)", ["systolic_bp", "diastolic_bp"]),
    ("Exposure available (adherence)", ["medication_adherence"]),
    ("Covariates available (age, sex, BMI, diabetes, smoking)", ["age", "sex", "bmi", "diabetes", "smoking_status"]),
]
analysis = clean.copy()
required = []
flow = []
for label, columns in flow_steps:
    required += columns
    n_before = len(analysis)
    analysis = analysis.dropna(subset=required)
    flow.append({"step": label, "n_remaining": len(analysis), "n_excluded": n_before - len(analysis)})
flow = pd.DataFrame(flow)
flow

## Exercise 2B (part 1): Who did we lose?

Compare patients **included** in the analysis set with those **excluded**, on age, diabetes and systolic BP. Use the `clean` data and mark who is in `analysis`.

In [ ]:
clean["included"] = ...    # True if the patient_id is in the analysis set

clean.groupby("included").agg(
    ...
).round(1)

**Write one limitation sentence** that links the missing data to its likely consequence (for selection, bias or precision).

**Your answer:** _Double-click here and write your answer._

In [ ]:
analysis = analysis.drop(columns=["hba1c_missing", "adherence_missing"], errors="ignore")
analysis.to_csv(DATA_PROCESSED / "analysis_set.csv", index=False, date_format="%Y-%m-%d")
flow.to_csv(OUT_TABLES / "analysis_set_flow.csv", index=False)
print("Saved analysis set:", analysis.shape)

# Part C: Univariate exploratory data analysis (block 4)

Univariate means one variable at a time. The right summary depends on the variable type:

| Variable type | Numeric summary | Graph |
|---|---|---|
| Continuous, roughly symmetric | Mean and standard deviation | Histogram |
| Continuous, skewed or with outliers | Median and interquartile range (IQR) | Histogram, boxplot |
| Categorical or binary | Count and percentage | Bar chart |

Always state the **denominator** (how many people the summary is based on).

## C1. Numeric variables

In [ ]:
numeric_vars = ["age", "bmi", "systolic_bp", "diastolic_bp", "hba1c", "physical_activity_min",
                "medication_adherence", "followup_visits_6m"]

def describe_numeric(data, columns):
    rows = []
    for col in columns:
        s = data[col].dropna()
        rows.append({"variable": col, "n": len(s), "mean": s.mean(), "sd": s.std(),
                     "median": s.median(), "q1": s.quantile(0.25), "q3": s.quantile(0.75),
                     "min": s.min(), "max": s.max()})
    return pd.DataFrame(rows).round(1)

numeric_summary = describe_numeric(analysis, numeric_vars)
numeric_summary

Notice that HbA1c has a smaller `n` than the other variables. It was not needed for the analysis set, so its missing values are still there. Always report the `n` next to a summary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
BRAND = "#002366"

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.histplot(analysis["systolic_bp"], bins=25, color=BRAND, ax=axes[0])
axes[0].set(title="Systolic blood pressure", xlabel="SBP (mmHg)", ylabel="Patients")
sns.histplot(analysis["medication_adherence"], bins=25, color=BRAND, ax=axes[1])
axes[1].set(title="Medication adherence", xlabel="Adherence score (%)", ylabel="Patients")
fig.tight_layout()
fig.savefig(OUT_FIGURES / "fig_univariate_sbp_adherence.png", dpi=300)
plt.show()

A boxplot is useful when a variable is skewed. Physical activity is a good example: most people do a little, a few do a lot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
sns.boxplot(x=analysis["physical_activity_min"], color="#DEE6EF", ax=ax)
ax.set(title="Weekly physical activity", xlabel="Minutes per week")
plt.show()
print("Mean:", round(analysis["physical_activity_min"].mean(), 1),
      "| Median:", analysis["physical_activity_min"].median())

## C2. Categorical variables

In [ ]:
def describe_categorical(data, columns):
    rows = []
    for col in columns:
        counts = data[col].value_counts(dropna=False)
        for level, n in counts.items():
            rows.append({"variable": col, "level": level, "n": n,
                         "percent": round(100 * n / len(data), 1)})
    return pd.DataFrame(rows)

categorical_summary = describe_categorical(analysis, ["sex", "clinic", "smoking_status", "diabetes", "uncontrolled_bp"])
categorical_summary

In [ ]:
order = ["Never", "Former", "Current"]
fig, ax = plt.subplots(figsize=(6, 3.6))
sns.countplot(x=analysis["smoking_status"], order=order, color=BRAND, ax=ax)
ax.set(title="Smoking status", xlabel="", ylabel="Patients")
plt.show()

## Exercise 2B (part 2): Your univariate evidence pack

Produce **one table and two figures** that describe BMI, clinic and the outcome `uncontrolled_bp`. Choose the summary and graph that suit each variable type. Save both figures to `outputs/figures`.

In [ ]:
# Table: summaries for BMI, clinic and uncontrolled_bp
...

# Figure 1
...

# Figure 2
...

**Write a five-sentence interpretation** of your evidence pack. Describe; do not explain causes.

**Your answer:** _Double-click here and write your answer._

## Key points

* Audit first, then standardise, validate, deduplicate and derive. Log every rule with a count.
* Let errors happen loudly (no silent coercion) and protect your rules with assertions.
* Missing data: describe how much, look for clues about why, choose an approach and state its assumption.
* A complete-case analysis is a transparent start, not a universal answer.
* Match the summary and the graph to the variable type, and always give the denominator.